# Quickstart

## 1. Load the LongMemEval dataset from the Hub

In [1]:
import os
from dotenv import find_dotenv, load_dotenv

# walks up from the cwd until it finds a .env
load_dotenv(find_dotenv(usecwd=True))
HF_TOKEN = os.environ["HUGGINGFACE_TOKEN"]

In [2]:
from huggingface_hub import hf_hub_download

DATA_PATH = hf_hub_download(
    repo_id="xiaowu0162/longmemeval-cleaned",
    filename="longmemeval_oracle.json",
    repo_type="dataset",
    token=HF_TOKEN,
)

DATA_PATH

/home/ssubrahmanya/summary-mem/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'/home/ssubrahmanya/.cache/huggingface/hub/datasets--xiaowu0162--longmemeval-cleaned/snapshots/98d7416c24c778c2fee6e6f3006e7a073259d48f/longmemeval_oracle.json'

In [3]:
import json

dataset = json.loads(open(DATA_PATH).read())
len(dataset)

500

## 2. Pick a random sample & describe it

In [4]:
import random

sample = random.choice(dataset)
sample["question_id"]

'945e3d21'

In [5]:
sessions = sample["haystack_sessions"]
turns_per_session = [len(s) for s in sessions]

print("question_id     :", sample["question_id"])
print("question_type   :", sample["question_type"])
print("question_date   :", sample.get("question_date"))
print("num_sessions    :", len(sessions))
print("turns/session   :", turns_per_session)
print("total_turns     :", sum(turns_per_session))
print("min/avg/max     :", min(turns_per_session), sum(turns_per_session) / len(sessions), max(turns_per_session))
print("question        :", sample["question"])
print("gold answer     :", sample["answer"])

question_id     : 945e3d21
question_type   : knowledge-update
question_date   : 2023/12/25 (Mon) 05:30
num_sessions    : 2
turns/session   : [12, 12]
total_turns     : 24
min/avg/max     : 12 12.0 12
question        : How often do I attend yoga classes to help with my anxiety?
gold answer     : Three times a week.


## 3. Run summary-mem on the sample

### 3.1 Load & instantiate

In [6]:
from summary_mem.clients import get_chat_client
from summary_mem.eval import MemoryEvaluator

client = get_chat_client()
evaluator = MemoryEvaluator(
    client,
    conversation_id=sample["question_id"],
    db_path="sample_memory.db",
)

### 3.2 Index the sessions into memory

In [7]:
evaluator.index(sample["haystack_sessions"], sample.get("haystack_dates"))
evaluator.memory.recall(sample["question_id"])

{'assistant': "The assistant is designed to help users prioritize their work tasks and support them in managing stress and mental health. They offer guidance on task prioritization using the Eisenhower Matrix, assisting users in categorizing tasks based on urgency and importance. The assistant encourages users to share their tasks and provides recommendations for handling them effectively, including the creation of detailed schedules.\n\nThe assistant emphasizes the importance of self-care and mindfulness practices. They suggest various stress-reducing apps and meditation techniques, such as mindfulness meditation, loving-kindness meditation, body scan, and 4-7-8 breathing, to help users manage anxiety and improve focus. The assistant also provides journaling prompts and reflection exercises for users exploring childhood trauma and emotional healing, stressing the significance of self-compassion, boundary setting, and seeking professional help.\n\nTo enhance users' understanding of anx

### 3.3 Query with the sample's question

In [8]:
result = evaluator.rag_qa(
    sample["question"],
    question_date=sample.get("question_date"),
    gold_answer=sample["answer"],
)

### 3.4 Compare against the gold answer

In [9]:
print("Q     :", result.question)
print("GOLD  :", result.gold)
print("MODEL :", result.answer)
print("CORRECT:", result.correct)

evaluator.close()

Q     : How often do I attend yoga classes to help with my anxiety?
GOLD  : Three times a week.
MODEL : You practice yoga three times a week to help clear your head and improve focus.
CORRECT: True


In [ ]:
from utils import load_accuracy_data

In [ ]:


candor = load_accuracy_data("candor", "big5_speaker_roberta-base_L11_ridgecv")
ds4ud_small = load_accuracy_data("2m2w", "big5_user_id_roberta-base_L11_ridgecv")


### DS4UD

In [2]:
from sqlalchemy import create_engine
import pandas as pd

DATABASE = "ssubrahmanya"

url = f"mysql://ssubrahmanya@localhost/{DATABASE}?charset=utf8mb4"
engine = create_engine(url, connect_args={"read_default_file": "~/.my.cnf"})

In [3]:
query = """
SELECT cohort, person_id, message_id, day_id, year_wave
FROM msg_essays_v9v11
"""

# msg_essays_v9v11 is a EMA level table
# message_id: {cohort}_{person_id}_{year_wave}_{day_id}_{ema_id}
message_table = pd.read_sql(query, engine)
display(message_table.head())

# read the outcome table
outcome_table = pd.read_sql("SELECT * FROM outcomes_v9v11_person", engine)
display(outcome_table.head())

,cohort,person_id,message_id,day_id,year_wave
0,v9,16,v9_16_y1_1_1,1,y1
1,v9,16,v9_16_y1_3_1,3,y1
2,v9,16,v9_16_y1_4_1,4,y1
3,v9,22,v9_22_y1_1_1,1,y1
4,v9,22,v9_22_y1_2_1,2,y1


,person_id,cohort,openness_score,conscientious_score,extravert_score,agreeable_score,neurotic_score
0,0,v9,NaN,NaN,NaN,NaN,NaN
1,1,v9,9.2,16.2,14.2,15.6,10.4
2,100,v9,NaN,NaN,NaN,NaN,NaN
3,1000,v9,NaN,NaN,NaN,NaN,NaN
4,1003,v9,NaN,NaN,NaN,NaN,NaN


In [4]:
(
    message_table.
    groupby(['cohort', 'person_id', 'year_wave'])["message_id"].
    size().
    reset_index(name='count').
    describe()
)

,count
count,879.000000
mean,16.467577
std,13.786479
min,1.000000
25%,5.000000
50%,11.000000
75%,30.500000
max,43.000000


In [13]:
# python dlatk/dlatkInterface.py -d ssubrahmanya -t msg_essays_v9v11 -c message_id --add_ngrams -n 1

#### Person-level personality assessment using `roberta-base`

In [24]:
from utils import load_accuracy_data

ds4ud = load_accuracy_data("ds4ud", "big5_person_id_roberta-base_L11_ridgecv")
(
    ds4ud[["outcome", "r", "r_p", "N"]]
    .rename(columns={"r_p": "p-value"})
    .round(2)
)

,outcome,r,p-value,N
0,agreeable_score,0.29,0.00,401.0
1,conscientious_score,0.30,0.00,401.0
2,extravert_score,0.13,0.01,401.0
3,neurotic_score,0.37,0.00,401.0
4,openness_score,0.14,0.00,401.0


#### LLM memory based personality assessment

##### In context memory - entire language history

In [ ]:
from utils import get_correlations

feature_table_name = "feat$bfi$ds4ud$person_id$ic$gpt4omini$s1"
results = get_correlations(engine, feature_table_name)[["trait", "r", "p", "CI.low", "CI.high"]]
display(results.round(2))

,trait,r,p,CI.low,CI.high
0,agreeableness,0.27,0.01,0.08,0.44
1,conscientiousness,0.34,0.00,0.16,0.51
2,extraversion,0.30,0.00,0.11,0.47
3,neuroticism,0.52,0.00,0.36,0.65
4,openness,0.08,0.42,-0.12,0.27


##### Most recent memory (20 language samples)

In [16]:
feature_table_name = "feat$bfi$ds4ud$person_id$ic$gpt4omini$rt20"
incontext = get_correlations(engine, feature_table_name)[["trait", "r", "p", "CI.low", "CI.high"]].set_index("trait")

feature_table_name = "feat$bfi$msg_essays_v9v11$person_id$sumplain$gpt4omini$rt20"
summary = get_correlations(engine, feature_table_name)[["trait", "r", "p", "CI.low", "CI.high"]].set_index("trait")

df = pd.concat({"in-context": incontext, "summary": summary}, axis=1)
display(df.reset_index().round(2))

trait in-context                      summary               \
                              r     p CI.low CI.high       r     p CI.low   
0      agreeableness       0.30  0.00   0.11    0.47    0.32  0.00   0.13   
1  conscientiousness       0.39  0.00   0.22    0.55    0.27  0.01   0.08   
2       extraversion       0.33  0.00   0.14    0.49    0.34  0.00   0.15   
3        neuroticism       0.56  0.00   0.40    0.68    0.50  0.00   0.34   
4           openness       0.15  0.14  -0.05    0.34    0.17  0.09  -0.03   

           
  CI.high  
0    0.48  
1    0.45  
2    0.50  
3    0.64  
4    0.36

##### Sporadic sampling (every 5th sample)

In [17]:
feature_table_name ="feat$bfi$ds4ud$person_id$ic$gpt4omini$s5"
incontext = get_correlations(engine, feature_table_name)[["trait", "r", "p", "CI.low", "CI.high"]].set_index("trait")

feature_table_name = "feat$bfi$msg_essays_v9v11$person_id$sumplain$gpt4omini$s5"
summary = get_correlations(engine, feature_table_name)[["trait", "r", "p", "CI.low", "CI.high"]].set_index("trait")

df = pd.concat({"in-context": incontext, "summary": summary}, axis=1)
display(df.round(2))

in-context                      summary                     
                           r     p CI.low CI.high       r     p CI.low CI.high
trait                                                                         
agreeableness           0.23  0.02   0.04    0.41    0.13  0.19  -0.07    0.32
conscientiousness       0.36  0.00   0.18    0.52    0.39  0.00   0.21    0.55
extraversion            0.35  0.00   0.17    0.51    0.25  0.01   0.06    0.43
neuroticism             0.46  0.00   0.30    0.61    0.46  0.00   0.29    0.60
openness                0.03  0.73  -0.16    0.23    0.12  0.23  -0.08    0.31